# 01 数据加载与初探

**目标**：读入 Online Retail 数据集，确认数据规模、字段类型和数据质量问题，为后续清洗与分析打基础。

**数据**：UCI Online Retail，2010-12-01 至 2011-12-09 的英国在线零售交易记录。

**运行前**：确认 `data/raw/Online Retail.xlsx` 存在，并在项目根目录启动 Jupyter。

In [1]:
from pathlib import Path
import time

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("matplotlib:", matplotlib.__version__)

pandas: 3.1.0
numpy: 2.5.4
matplotlib: 3.11.2


In [2]:
ROOT = Path.cwd()
while not (ROOT / "data").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

DATA_PATH = ROOT / "data" / "raw" / "Online Retail.xlsx"
print("项目根目录:", ROOT)
print("数据文件:", DATA_PATH)
assert DATA_PATH.exists(), "找不到数据文件，请先下载并放到 data/raw/"

项目根目录: D:\ds-portfolio\retail-analysis
数据文件: D:\ds-portfolio\retail-analysis\data\raw\Online Retail.xlsx


In [3]:
start = time.time()
df = pd.read_excel(DATA_PATH)
elapsed = time.time() - start

print(f"读取完成，用时 {elapsed:.1f} 秒")
print(f"数据形状: {df.shape[0]:,} 行 × {df.shape[1]} 列")

读取完成，用时 20.2 秒
数据形状: 541,909 行 × 8 列


In [4]:
print("列名:")
print(list(df.columns))
print("\n字段类型:")
print(df.dtypes)

列名:
['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']

字段类型:
InvoiceNo              object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[us]
UnitPrice             float64
CustomerID            float64
Country                   str
dtype: object


In [5]:
df.head(10)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
5,536365,22752,SET 7 BABUSHKA NESTING BOXES,2,2010-12-01 08:26:00,7.65,17850.0,United Kingdom
6,536365,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,2010-12-01 08:26:00,4.25,17850.0,United Kingdom
7,536366,22633,HAND WARMER UNION JACK,6,2010-12-01 08:28:00,1.85,17850.0,United Kingdom
8,536366,22632,HAND WARMER RED POLKA DOT,6,2010-12-01 08:28:00,1.85,17850.0,United Kingdom
9,536367,84879,ASSORTED COLOUR BIRD ORNAMENT,32,2010-12-01 08:34:00,1.69,13047.0,United Kingdom


In [6]:
quality = pd.DataFrame({
    "缺失数量": df.isna().sum(),
    "缺失比例(%)": (df.isna().mean() * 100).round(2),
})
print(quality)
print(f"\n完全重复的行数: {df.duplicated().sum():,}")

               缺失数量  缺失比例(%)
InvoiceNo         0     0.00
StockCode         0     0.00
Description    1454     0.27
Quantity          0     0.00
InvoiceDate       0     0.00
UnitPrice         0     0.00
CustomerID   135080    24.93
Country           0     0.00



完全重复的行数: 5,268


In [7]:
print(f"日期范围: {df['InvoiceDate'].min()} → {df['InvoiceDate'].max()}")
print(f"订单数（去重 InvoiceNo）: {df['InvoiceNo'].nunique():,}")
print(f"商品数（去重 StockCode）: {df['StockCode'].nunique():,}")
print(f"客户数（去重 CustomerID）: {df['CustomerID'].nunique():,}")
print(f"国家数: {df['Country'].nunique()}")
print(f"数量为负的记录数: {(df['Quantity'] < 0).sum():,}")
print(f"单价小于等于 0 的记录数: {(df['UnitPrice'] <= 0).sum():,}")

日期范围: 2010-12-01 08:26:00 → 2011-12-09 12:50:00
订单数（去重 InvoiceNo）: 25,900
商品数（去重 StockCode）: 4,070
客户数（去重 CustomerID）: 4,372
国家数: 38
数量为负的记录数: 10,624
单价小于等于 0 的记录数: 2,517


In [8]:
df["Revenue"] = df["Quantity"] * df["UnitPrice"]
valid = df[(df["Quantity"] > 0) & (df["UnitPrice"] > 0)]

print(f"全部记录销售额合计: {df['Revenue'].sum():,.2f}")
print(f"有效记录销售额合计: {valid['Revenue'].sum():,.2f}")
print("\n销售额 Top 10 国家:")
print(
    valid.groupby("Country")["Revenue"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .round(2)
)

全部记录销售额合计: 9,747,747.93


有效记录销售额合计: 10,666,684.54

销售额 Top 10 国家:
Country
United Kingdom    9025222.08
Netherlands        285446.34
EIRE               283453.96
Germany            228867.14
France             209715.11
Australia          138521.31
Spain               61577.11
Switzerland         57089.90
Belgium             41196.34
Sweden              38378.33
Name: Revenue, dtype: float64


## 初步结论

1. **数据规模**：541,909 行 × 8 列，时间范围 2010-12-01 至 2011-12-09，约一年；去重 `InvoiceNo` 25,900 个（含取消发票），商品编码 4,070 个，国家/地区 38 个。
2. **数据质量**：
   - `CustomerID` 缺失 135,080 行（24.93%），对应匿名交易；匿名交易约占全部销售额的 14.85%，客户级分析需要单独处理。
   - `Description` 缺失 1,454 行（0.27%）。
   - 完全重复行 5,268 行，清洗时需要去重。
3. **退货/取消**：数量为负的记录 10,624 行，其中 9,288 行来自 `C` 开头的取消发票，合计 -896,812.49；另有 1,336 行负数量不是 `C` 发票，可能是库存调整或人工调整，需要单独核查。
4. **销售结构**：有效销售额 10,666,684.54；英国贡献 84.61%，业务高度依赖本土市场。有有效购买的客户 4,338 个，Top 10% 客户贡献 61.33% 的销售额，客户集中度高。
5. **商品口径提醒**：销售额 Top 1 的 `DOTCOM POSTAGE` 是邮费，不是商品；做商品分析时要排除 `POST`、`PADS`、`D`、`M` 等非商品编码，避免结论失真。
6. **下一步清洗**：
   - 去重；
   - 区分正常销售、取消退货、库存调整；
   - 商品分析排除非商品编码；
   - 客户级分析排除 `CustomerID` 缺失；
   - 检查 `UnitPrice <= 0` 的 2,517 行记录。
